# V035 verified results

One larger recurrent architecture under the same objective, features, data and schedule. Native mean MAE controls promotion; median is secondary. A improvement requires cold grouped confirmation before final fitting.

In [1]:
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
P=Path.cwd();O=P/'outputs/controlled_experiments_v035';R=O/'results_review';s=json.loads((O/'screen_summary.json').read_text());a=json.loads((P/'research/screen_review_v035.json').read_text());assert a['status']=='passed' and a['screen_summary_sha256']==hashlib.sha256((O/'screen_summary.json').read_bytes()).hexdigest()
print('Completed original-A comparisons (mean MAE):')
print(pd.DataFrame([{k:r[k] for k in ['name','MAE','gain','eligible']} for r in s['candidates']]).to_string(index=False))
report=json.loads((O/'capacity_laplace_curve/report.json').read_text());old=json.loads((P/'outputs/controlled_experiments_v029/laplace_curve/report.json').read_text());h=report['epochs'];oh=old['epochs']
fig,ax=plt.subplots(figsize=(10,5));ax.plot([e['epoch'] for e in h],[e['validation_inspiratory_mae'] for e in h],label='Larger LSTM validation',color='#216e9a');ax.plot([e['epoch'] for e in oh],[e['validation_inspiratory_mae'] for e in oh],label='Previous LSTM validation',color='#c45a36');ax.plot([e['epoch'] for e in h],[e['training_eval_inspiratory_mae'] for e in h],label='Larger LSTM training diagnostic',color='#216e9a',linestyle='--');ax.set(xlabel='Epoch (same split, data size and optimizer steps)',ylabel='Native inspiratory mean MAE',title='V035 capacity comparison — same objective, fixed 200-epoch schedule',ylim=(0,.65));ax.legend();fig.tight_layout();fig.savefig(R/'learning_curves.png',dpi=160);plt.close(fig)
df=pd.read_csv(P/'data/train.csv',usecols=['id','breath_id','R','C','u_out','pressure'],dtype={'pressure':'float32'});bids=df.breath_id.to_numpy().reshape(-1,80)[:,0];split=np.load(P/'outputs/improvements_v005/sequence/split_breath_ids.npz');val=np.flatnonzero(np.isin(bids,split['A_validation']));phase=df.u_out.to_numpy().reshape(-1,80)[val];mask=phase==0;ids=df.id.to_numpy().reshape(-1,80)[val][mask];assert np.array_equal(ids,np.load(O/'A_row_ids.npy'));truth=df.pressure.to_numpy().reshape(-1,80)[val][mask].astype(np.float64);rc=df[['R','C']].to_numpy().reshape(-1,80,2)[val];point_rc=rc[mask];base=np.load(P/'outputs/postprocessing_screen_v016/weighted_median_midpoint_then_nearest_training_pressure_codebook_A_predictions.npy');candidate=np.load(P/s['selected_for_confirmation']['prediction_file']);be=abs(base-truth);ce=abs(candidate-truth);table=[]
for setting in sorted(set(map(tuple,point_rc))):
 z=np.all(point_rc==setting,axis=1);table.append(dict(R=int(setting[0]),C=int(setting[1]),points=int(z.sum()),baseline_MAE=float(be[z].mean()),candidate_MAE=float(ce[z].mean()),baseline_total_error=float(be[z].sum()),candidate_total_error=float(ce[z].sum()),total_error_reduction=float((be[z]-ce[z]).sum())))
table=pd.DataFrame(table);table.to_csv(R/'ensemble_error_by_RC.csv',index=False);print(table.to_string(index=False));order=table.sort_values('baseline_total_error',ascending=False);labels=['R='+str(int(r.R))+', C='+str(int(r.C)) for r in order.itertuples()]
fig,axes=plt.subplots(1,2,figsize=(13,5));axes[0].barh(labels,order.baseline_total_error,label='Accepted baseline',alpha=.6);axes[0].barh(labels,order.candidate_total_error,label='V035 ensemble',alpha=.6);axes[0].invert_yaxis();axes[0].set(title='Groups contributing most total error',xlabel='Total absolute pressure error');axes[0].legend();axes[1].barh(labels,order.total_error_reduction,color=np.where(order.total_error_reduction>0,'#34816a','#b4584e'));axes[1].invert_yaxis();axes[1].axvline(0,color='black',linewidth=.8);axes[1].set(title='Contribution to improvement',xlabel='Baseline minus candidate total error');fig.suptitle('Original grouped A validation; no F result exists');fig.tight_layout();fig.savefig(R/'ensemble_error_contributions.png',dpi=160);plt.close(fig)
print('These are validation results, not a new official score. Cold F confirmation and final all-data fit have not run.')


Matplotlib created a temporary cache directory at /var/folders/hx/8sx7grc130g7cssms36qx6rh0000gn/T/matplotlib-uigwsl4b because the default path (/Users/generalclasher/.matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Completed original-A comparisons (mean MAE):
                           name      MAE     gain  eligible
   capacity_laplace_replacement 0.177529 0.005393      True
capacity_laplace_retained_curve 0.178721 0.004201      True
 R  C  points  baseline_MAE  candidate_MAE  baseline_total_error  candidate_total_error  total_error_reduction
 5 10   22461      0.174696       0.165851           3923.844181            3725.170259             198.673922
 5 20   23070      0.116037       0.111743           2676.965191            2577.909529              99.055662
 5 50   22061      0.124068       0.122822      